In [ ]:
import os
import torch
from d2l import torch as d2l

# 配置数据集的下载信息
# 'fra-eng' 是英语-法语平行语料库，包含英法句子对
d2l.DATA_HUB['fra-eng'] = (d2l.DATA_URL + 'fra-eng.zip',
                           '94646ad1522d915e7b0f9296181140edcf86a4f5')

def read_data_nmt():
    """载入"英语-法语"数据集
    
    NMT: Neural Machine Translation（神经机器翻译）
    
    返回:
        str: 包含英法句子对的原始文本，每行一个句子对，格式为 "英文\t法文"
        
    功能说明:
        1. 自动下载并解压数据集（如果本地没有）
        2. 读取 fra.txt 文件内容
        3. 返回原始文本用于后续处理
    """
    # 下载并解压数据集到本地目录
    data_dir = d2l.download_extract('fra-eng')
    
    # 打开并读取数据文件
    with open(os.path.join(data_dir, 'fra.txt'), 'r',
             encoding='utf-8') as f:
        return f.read()

# 读取原始数据
raw_text = read_data_nmt()

# 打印前75个字符，查看数据格式
# 预期格式：Go.\tVa !\n（英文句子\t法文句子\n）
print(raw_text[:75])


In [ ]:
def preprocess_nmt(text):
    """预处理"英语-法语"数据集
    
    参数:
        text: 原始文本数据
        
    返回:
        str: 预处理后的文本
        
    预处理步骤：
        1. 替换特殊空格字符为标准空格
        2. 将所有字母转换为小写（统一大小写，减少词表大小）
        3. 在标点符号前添加空格（方便后续分词）
    """
    def no_space(char, prev_char):
        """判断字符前是否不需要空格
        
        参数:
            char: 当前字符
            prev_char: 前一个字符
            
        返回:
            bool: 如果当前字符是标点且前一个字符不是空格，返回True
        """
        return char in set(',.!?') and prev_char != ' '

    # 步骤1: 替换特殊空格字符
    # \u202f 是窄不换行空格，\xa0 是不换行空格
    text = text.replace('\u202f', ' ').replace('\xa0', ' ').lower()
    
    # 步骤2: 在标点符号前插入空格
    # 这样可以将 "Hello,world" 变成 "hello , world"，便于分词
    out = [' ' + char if i > 0 and no_space(char, text[i - 1]) else char
           for i, char in enumerate(text)]
    
    return ''.join(out)

# 预处理文本
text = preprocess_nmt(raw_text)

# 打印前80个字符，观察预处理效果
# 预期效果：大写变小写，标点前有空格
print(text[:80])


In [ ]:
def tokenize_nmt(text, num_examples=None):
    """词元化"英语-法语"数据集
    
    词元化（Tokenization）：将文本分割成最小的语义单位（词元/token）
    
    参数:
        text: 预处理后的文本数据
        num_examples: 只处理前 num_examples 个样本，None 表示处理全部
        
    返回:
        source: 源语言（英语）词元列表的列表，如 [['go', '.'], ['hi', '.']]
        target: 目标语言（法语）词元列表的列表，如 [['va', '!'], ['salut', '!']]
        
    功能说明:
        1. 按换行符分割文本，得到每一行（一个句子对）
        2. 按制表符分割每行，得到源句子和目标句子
        3. 按空格分割句子，得到词元列表
    """
    source, target = [], []
    
    # 遍历每一行文本
    for i, line in enumerate(text.split('\n')):
        # 如果指定了样本数量限制，超过后停止处理
        if num_examples and i > num_examples:
            break
        
        # 按制表符分割，得到英文和法文句子
        parts = line.split('\t')
        
        # 确保这一行包含两部分（英文和法文）
        if len(parts) == 2:
            # 按空格分割成词元，并添加到对应列表
            source.append(parts[0].split(' '))  # 英文句子的词元列表
            target.append(parts[1].split(' '))  # 法文句子的词元列表
    
    return source, target

# 词元化处理
source, target = tokenize_nmt(text)

# 查看前6个样本的词元化结果
# 每个元素是一个词元列表，如 ['go', '.']
source[:6], target[:6]


In [ ]:
def show_list_len_pair_hist(legend, xlabel, ylabel, xlist, ylist):
    """绘制列表长度对的直方图
    
    用于可视化源语言和目标语言句子的长度分布，帮助我们了解：
    - 句子长度的分布情况
    - 源语言和目标语言的长度差异
    - 是否需要截断或填充
    
    参数:
        legend: 图例标签列表，如 ['source', 'target']
        xlabel: x轴标签，通常是 "每个序列的词元数"
        ylabel: y轴标签，通常是 "数量"
        xlist: 源语言句子列表
        ylist: 目标语言句子列表
    """
    # 设置图形大小
    d2l.set_figsize()
    
    # 绘制直方图
    # [len(l) for l in xlist] 计算每个源句子的长度
    # [len(l) for l in ylist] 计算每个目标句子的长度
    _, _, patches = d2l.plt.hist(
        [[len(l) for l in xlist], [len(l) for l in ylist]])
    
    # 设置坐标轴标签
    d2l.plt.xlabel(xlabel)
    d2l.plt.ylabel(ylabel)
    
    # 为目标语言的直方图添加斜线纹理，以区分两种语言
    for patch in patches[1].patches:
        patch.set_hatch('/')
    
    # 添加图例
    d2l.plt.legend(legend)

# 绘制源语言和目标语言的句子长度分布直方图
# 这可以帮助我们了解数据特征，决定合适的序列长度
show_list_len_pair_hist(['source', 'target'], '# tokens per sequence',
                        'count', source, target);


In [ ]:
# 构建源语言（英语）的词表
# 词表（Vocabulary）：将词元映射到整数索引的字典
src_vocab = d2l.Vocab(source, min_freq=2,
                      reserved_tokens=['<pad>', '<bos>', '<eos>'])

# 参数说明：
# - source: 源语言的词元列表
# - min_freq=2: 只保留出现至少2次的词，过滤低频词以减小词表大小
# - reserved_tokens: 保留的特殊词元
#   - '<pad>': padding，用于填充较短的序列，使批次中所有序列长度相同
#   - '<bos>': beginning of sequence，序列开始标记
#   - '<eos>': end of sequence，序列结束标记

# 输出词表大小
# 词表大小 = 有效词元数 + 特殊词元数 + 未知词元('<unk>')
len(src_vocab)


In [ ]:
def truncate_pad(line, num_steps, padding_token):
    """截断或填充文本序列，使其长度固定为 num_steps
    
    在深度学习中，同一批次的序列必须有相同的长度，因此需要：
    - 截断：如果序列太长，只保留前 num_steps 个词元
    - 填充：如果序列太短，用 padding_token 填充到 num_steps
    
    参数:
        line: 词元索引列表，如 [3, 5, 12, 8]
        num_steps: 目标序列长度
        padding_token: 填充词元的索引，通常是 '<pad>' 对应的索引
        
    返回:
        长度为 num_steps 的词元索引列表
        
    示例:
        如果 line = [3, 5, 12]，num_steps = 10，padding_token = 1
        返回 [3, 5, 12, 1, 1, 1, 1, 1, 1, 1]
    """
    if len(line) > num_steps:
        # 序列太长，截断到 num_steps
        return line[:num_steps]
    
    # 序列太短或刚好，填充到 num_steps
    # 填充数量 = num_steps - 当前长度
    return line + [padding_token] * (num_steps - len(line))

# 测试截断填充函数
# 将第一个源句子的词元索引填充到长度10
truncate_pad(src_vocab[source[0]], 10, src_vocab['<pad>'])


In [ ]:
def build_array_nmt(lines, vocab, num_steps):
    """将机器翻译的文本序列转换成小批量张量
    
    将词元列表转换为模型可以处理的张量格式，包括：
    1. 词元 -> 索引：使用词表将词元转换为整数索引
    2. 添加结束标记：在每个序列末尾添加 '<eos>'
    3. 截断填充：统一序列长度
    4. 计算有效长度：记录每个序列的真实长度（不含填充）
    
    参数:
        lines: 词元列表的列表，如 [['go', '.'], ['hi', '.']]
        vocab: 词表对象，用于将词元转换为索引
        num_steps: 固定序列长度
        
    返回:
        array: 形状为 (num_sequences, num_steps) 的张量，包含词元索引
        valid_len: 形状为 (num_sequences,) 的张量，记录每个序列的有效长度
        
    示例:
        输入: [['go', '.'], ['hi', '.']]
        输出: tensor([[3, 5, 2, 1, 1],    # 3=go, 5=., 2=<eos>, 1=<pad>
                     [7, 5, 2, 1, 1]])   # 7=hi, 5=., 2=<eos>, 1=<pad>
              tensor([3, 3])              # 两个句子的有效长度都是3
    """
    # 步骤1: 将词元转换为索引
    # vocab[l] 将词元列表 l 转换为索引列表
    lines = [vocab[l] for l in lines]
    
    # 步骤2: 在每个序列末尾添加 '<eos>' 结束标记
    # 这样模型就知道句子在哪里结束
    lines = [l + [vocab['<eos>']] for l in lines]
    
    # 步骤3: 截断或填充到固定长度，并转换为张量
    array = torch.tensor([truncate_pad(
        l, num_steps, vocab['<pad>']) for l in lines])
    
    # 步骤4: 计算每个序列的有效长度（排除填充词元）
    # array != vocab['<pad>'] 生成布尔张量，True表示非填充词元
    # .sum(1) 按行求和，得到每行的非填充词元数量
    valid_len = (array != vocab['<pad>']).type(torch.int32).sum(1)
    
    return array, valid_len


In [ ]:
def load_data_nmt(batch_size, num_steps, num_examples=600):
    """返回翻译数据集的迭代器和词表
    
    这是一个完整的数据加载函数，整合了所有数据处理步骤：
    1. 读取原始文本
    2. 预处理文本
    3. 词元化
    4. 构建词表
    5. 转换为张量
    6. 创建数据迭代器
    
    参数:
        batch_size: 批次大小，每次训练使用的样本数
        num_steps: 序列的固定长度
        num_examples: 使用的样本数量，默认600个（用于快速测试）
        
    返回:
        data_iter: 数据迭代器，每次迭代返回一个批次的数据
        src_vocab: 源语言词表
        tgt_vocab: 目标语言词表
        
    数据迭代器返回的每个批次包含：
        - src_array: 源序列张量，形状 (batch_size, num_steps)
        - src_valid_len: 源序列有效长度，形状 (batch_size,)
        - tgt_array: 目标序列张量，形状 (batch_size, num_steps)
        - tgt_valid_len: 目标序列有效长度，形状 (batch_size,)
    """
    # 步骤1-3: 读取、预处理、词元化
    text = preprocess_nmt(read_data_nmt())
    source, target = tokenize_nmt(text, num_examples)
    
    # 步骤4: 构建源语言和目标语言的词表
    src_vocab = d2l.Vocab(source, min_freq=2,
                          reserved_tokens=['<pad>', '<bos>', '<eos>'])
    tgt_vocab = d2l.Vocab(target, min_freq=2,
                          reserved_tokens=['<pad>', '<bos>', '<eos>'])
    
    # 步骤5: 将词元列表转换为张量，并计算有效长度
    src_array, src_valid_len = build_array_nmt(source, src_vocab, num_steps)
    tgt_array, tgt_valid_len = build_array_nmt(target, tgt_vocab, num_steps)
    
    # 步骤6: 将所有数据组合，并创建数据迭代器
    data_arrays = (src_array, src_valid_len, tgt_array, tgt_valid_len)
    data_iter = d2l.load_array(data_arrays, batch_size)
    
    return data_iter, src_vocab, tgt_vocab


In [ ]:
# 创建训练数据迭代器和词表
train_iter, src_vocab, tgt_vocab = load_data_nmt(batch_size=2, num_steps=8)

# 参数说明：
# - batch_size=2: 每个批次包含2个样本（句子对）
# - num_steps=8: 每个序列的固定长度为8个词元

# 从迭代器中取出第一个批次，查看数据格式
for X, X_valid_len, Y, Y_valid_len in train_iter:
    # X: 源序列（英语），形状 (2, 8)，包含词元索引
    print('X:', X.type(torch.int32))
    
    # X_valid_len: 源序列的有效长度，形状 (2,)
    # 表示每个句子的真实长度（不包括填充）
    print('X的有效长度:', X_valid_len)
    
    # Y: 目标序列（法语），形状 (2, 8)，包含词元索引
    print('Y:', Y.type(torch.int32))
    
    # Y_valid_len: 目标序列的有效长度，形状 (2,)
    print('Y的有效长度:', Y_valid_len)
    
    # 只查看第一个批次就退出
    break

# 输出说明：
# - 数字是词元在词表中的索引
# - 较小的索引通常是特殊词元（如 0='<unk>', 1='<pad>', 2='<bos>', 3='<eos>'）
# - 较大的索引是实际的词元（如单词）
# - 有效长度告诉我们序列中有多少个真实词元，其余是填充
